# Laboratorio 4: análisis de datos geoespaciales — lagos Atitlán y Amatitlán

Este notebook implementa el flujo completo de adquisición y análisis para los lagos Atitlán y Amatitlán usando Sentinel-2 L2A mediante openEO. Se utilizan únicamente las fechas oficiales de la guía.

**Ejercicios cubiertos:** 1) conexión a la API, 2) obtención de datos, 3) cálculo de NDVI, NDWI y NDCI para cianobacterias, 4) análisis temporal (promedio por lago y fecha), 5) análisis espacial (mapas interactivos y comparativos), 6) correlación de NDVI/NDWI con la cianobacteria, 7) comparación entre lagos, y 8) análisis exploratorio adicional (extensión espacial, zonas persistentes, distribución entre fechas y patrón estacional).

El NDCI es el índice cuantitativo de cianobacterias utilizado en el script de Sentinel Hub: `(B05 - B04) / (B05 + B04)`. Se reporta como indicador espectral de clorofila-a/cianobacterias y no como una medición de laboratorio.

## 0. Dependencias y configuración

Si alguna dependencia no está instalada, ejecute `pip install openeo rasterio geopandas shapely numpy pandas matplotlib` en el entorno del notebook. La autenticación OIDC se realiza en el navegador y requiere una cuenta de Copernicus Data Space.

In [1]:
from datetime import date, timedelta
from pathlib import Path
import warnings

import folium
import geopandas as gpd
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import openeo
import pandas as pd
import rasterio
from branca.colormap import LinearColormap
from rasterio.features import geometry_mask
from rasterio.transform import array_bounds
from rasterio.warp import Resampling as WarpResampling
from rasterio.warp import calculate_default_transform, reproject
from shapely.geometry import box

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 30)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data" / "GIS" / "lab4_avance"
DATA_DIR.mkdir(parents=True, exist_ok=True)
API_URL = "https://openeo.dataspace.copernicus.eu"
COLLECTION = "SENTINEL2_L2A"
# Las bandas se solicitan en el mismo orden en que se interpretan localmente.
BANDS = ["B02", "B03", "B04", "B05", "B08", "SCL"]
NDCI_HIGH_THRESHOLD = 0.20
print(f"Datos de salida: {DATA_DIR.resolve()}")

/Users/dijan/Documents/U/Data Science/Lab4/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Datos de salida: /Users/dijan/Documents/U/Data Science/Lab4/data/GIS/lab4_avance


## 1. Áreas y fechas oficiales

Las extensiones corresponden a la guía del laboratorio. Para cada fecha se solicita una ventana de un día, porque openEO espera un intervalo temporal y la fecha final es exclusiva en muchos backends.

In [2]:
LAKES = {
    "Atitlan": {
        "west": -91.326256, "east": -91.07151,
        "south": 14.5948, "north": 14.750979,
    },
    "Amatitlan": {
        "west": -90.638065, "east": -90.512924,
        "south": 14.412347, "north": 14.493799,
    },
}

OFFICIAL_DATES = {
    "Atitlan": [
        "2025-01-18", "2025-04-13", "2025-05-13",
        "2025-07-17", "2025-11-21", "2025-12-29",
        "2026-02-12", "2026-03-24", "2026-04-13",
        "2026-04-28", "2026-07-22",
    ],
    "Amatitlan": [
        "2025-01-28", "2025-04-15", "2025-04-28",
        "2025-11-24", "2026-01-08", "2026-02-02",
        "2026-02-07", "2026-03-29", "2026-04-13",
        "2026-04-28", "2026-06-19",
    ],
}

assert set(LAKES) == set(OFFICIAL_DATES)
assert all(len(v) == 11 for v in OFFICIAL_DATES.values())
print("Configuración validada: 2 lagos y 11 fechas oficiales por lago.")

Configuración validada: 2 lagos y 11 fechas oficiales por lago.


## 2. Geometrías de trabajo

Si existe un GeoJSON proporcionado por el curso, se utiliza para calcular estadísticas solamente sobre el cuerpo de agua. Como respaldo reproducible, el notebook construye un polígono con el bounding box oficial. El respaldo permite ejecutar el flujo, pero para la entrega final se debe preferir el GeoJSON, porque el bounding box incluye tierra.

In [3]:
def find_lake_geometry(lake_name, extent):
    candidates = [
        p for p in list(BASE_DIR.rglob("*.geojson")) + list(BASE_DIR.rglob("*.json"))
        if not any(part.startswith(".") or part in ("data",) for part in p.relative_to(BASE_DIR).parts[:-1])
        and lake_name.lower() in p.name.lower()
    ]
    for path in candidates:
        try:
            frame = gpd.read_file(path)
            if frame.empty or frame.geometry.isna().any() or frame.geometry.is_empty.any():
                continue
            frame = frame.to_crs("EPSG:4326")
            return frame[["geometry"]], f"GeoJSON: {path.name}"
        except Exception:
            continue
    fallback = gpd.GeoDataFrame(
        {"geometry": [box(extent["west"], extent["south"], extent["east"], extent["north"])]},
        crs="EPSG:4326",
    )
    return fallback, "Bounding box oficial (respaldo)"

GEOMETRIES = {}
for lake_name, extent in LAKES.items():
    GEOMETRIES[lake_name], source = find_lake_geometry(lake_name, extent)
    print(f"{lake_name}: {source}")

Atitlan: Bounding box oficial (respaldo)


Amatitlan: Bounding box oficial (respaldo)


## Ejercicio 1. Conexión con openEO

La siguiente celda abre la conexión y ejecuta OIDC. No se guardan credenciales en el notebook.

In [4]:
connection = openeo.connect(API_URL)
connection = connection.authenticate_oidc()
print(f"Conexión autenticada: {API_URL}")
print("Backend: ", connection.describe_account())

Authenticated using refresh token.
Conexión autenticada: https://openeo.dataspace.copernicus.eu


Backend:  {'info': {'oidc_userinfo': {'email': 'pad23663@uvg.edu.gt', 'email_verified': True, 'family_name': 'Padilla', 'given_name': 'Luis', 'name': 'Luis Padilla', 'preferred_username': 'pad23663@uvg.edu.gt', 'sub': '6d5c54ba-eb01-45fb-b63a-42c48bbcf64c'}}, 'name': 'Luis Padilla', 'user_id': '6d5c54ba-eb01-45fb-b63a-42c48bbcf64c'}


## Ejercicio 2. Descarga de escenas Sentinel-2 L2A

Se descarga una escena por lago y fecha. Se incluyen B02-B05 y B08 para los índices, y SCL para excluir píxeles de nubes, sombras y datos inválidos. Los GeoTIFF se conservan localmente para que el análisis sea reproducible sin volver a consultar la API.

In [5]:
def scene_path(lake_name, date_text):
    return DATA_DIR / f"{lake_name}_{date_text}.tif"

def download_scene(connection, lake_name, date_text):
    target = scene_path(lake_name, date_text)
    if target.exists() and target.stat().st_size > 0:
        return target
    start = date.fromisoformat(date_text)
    end = start + timedelta(days=1)
    cube = connection.load_collection(
        COLLECTION,
        spatial_extent=LAKES[lake_name],
        temporal_extent=[start.isoformat(), end.isoformat()],
        bands=BANDS,
    )
    job = connection.create_job(
        cube.save_result(format="GTIFF"),
        title=f"Lab4_{lake_name}_{date_text}",
    )
    job.start_and_wait()
    job.download_results(str(target))
    if not target.exists():
        raise FileNotFoundError(f"openEO no creó {target}")
    return target

scene_files = []
for lake_name in LAKES:
    for date_text in OFFICIAL_DATES[lake_name]:
        path = download_scene(connection, lake_name, date_text)
        scene_files.append((lake_name, date_text, path))
        print(f"Disponible: {path.name}")
print(f"Escenas disponibles: {len(scene_files)}")

Disponible: Atitlan_2025-01-18.tif
Disponible: Atitlan_2025-04-13.tif
Disponible: Atitlan_2025-05-13.tif
Disponible: Atitlan_2025-07-17.tif
Disponible: Atitlan_2025-11-21.tif
Disponible: Atitlan_2025-12-29.tif
Disponible: Atitlan_2026-02-12.tif
Disponible: Atitlan_2026-03-24.tif
Disponible: Atitlan_2026-04-13.tif
Disponible: Atitlan_2026-04-28.tif
Disponible: Atitlan_2026-07-22.tif
Disponible: Amatitlan_2025-01-28.tif
Disponible: Amatitlan_2025-04-15.tif
Disponible: Amatitlan_2025-04-28.tif
Disponible: Amatitlan_2025-11-24.tif
Disponible: Amatitlan_2026-01-08.tif
Disponible: Amatitlan_2026-02-02.tif
Disponible: Amatitlan_2026-02-07.tif
Disponible: Amatitlan_2026-03-29.tif
Disponible: Amatitlan_2026-04-13.tif
Disponible: Amatitlan_2026-04-28.tif
Disponible: Amatitlan_2026-06-19.tif
Escenas disponibles: 22
